In [1]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path("../data/raw")

file = DATA_DIR / "Academic_Project_Similarity_Synthetic_Dataset.xlsx"

projects = pd.read_excel(
    file,
    sheet_name="Projects"
)

pairs = pd.read_excel(
    file,
    sheet_name="SimilarityPairs"
)

In [2]:
sections = {
    "title": "project_title",
    "abstract": "abstract",
    "problem": "problem_statement",
    "objectives": "objectives",
    "methodology": "proposed_methodology",
    "dataset": "dataset",
    "algorithm": "algorithm_model",
    "technology": "technology_tools"
}

In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizers = {}
tfidf_matrices = {}

for section_name, column in sections.items():

    vectorizer = TfidfVectorizer(
        stop_words="english",
        ngram_range=(1, 2)
    )

    matrix = vectorizer.fit_transform(
        projects[column].fillna("").astype(str)
    )

    vectorizers[section_name] = vectorizer
    tfidf_matrices[section_name] = matrix

    print(
        section_name,
        matrix.shape
    )

title (600, 477)
abstract (600, 454)
problem (600, 199)
objectives (600, 188)
methodology (600, 227)
dataset (600, 83)
algorithm (600, 19)
technology (600, 89)


In [4]:
project_to_index = {
    project_id: index
    for index, project_id
    in enumerate(projects["project_id"])
}

In [5]:
from sklearn.metrics.pairwise import cosine_similarity

for section_name in sections:

    scores = []

    matrix = tfidf_matrices[section_name]

    for _, row in pairs.iterrows():

        idx_a = project_to_index[row["project_A_id"]]
        idx_b = project_to_index[row["project_B_id"]]

        score = cosine_similarity(
            matrix[idx_a],
            matrix[idx_b]
        )[0][0]

        scores.append(score)

    pairs[f"{section_name}_similarity"] = scores

In [6]:
display(
    pairs[
        [
            "expert_similarity_label",
            "title_similarity",
            "abstract_similarity",
            "problem_similarity",
            "objectives_similarity",
            "methodology_similarity",
            "dataset_similarity",
            "algorithm_similarity",
            "technology_similarity"
        ]
    ].head()
)

,expert_similarity_label,title_similarity,abstract_similarity,problem_similarity,objectives_similarity,methodology_similarity,dataset_similarity,algorithm_similarity,technology_similarity
0,0,0.006180,0.224976,0.391457,0.325653,0.018385,0.000000,0.0,0.215376
1,0,0.004200,0.179554,0.233032,0.204741,0.054619,0.111653,0.0,0.017201
2,0,0.005641,0.189573,0.356987,0.319630,0.000000,0.000000,0.0,0.019750
3,0,0.060742,0.150121,0.297045,0.284560,0.016518,0.050648,0.0,0.024160
4,3,0.491623,0.718970,1.000000,1.000000,0.040283,1.000000,1.0,0.022558


In [7]:
section_columns = [
    "title_similarity",
    "abstract_similarity",
    "problem_similarity",
    "objectives_similarity",
    "methodology_similarity",
    "dataset_similarity",
    "algorithm_similarity",
    "technology_similarity"
]

section_means = (
    pairs
    .groupby("expert_similarity_label")[section_columns]
    .mean()
)

display(section_means)

,title_similarity,abstract_similarity,problem_similarity,objectives_similarity,methodology_similarity,dataset_similarity,algorithm_similarity,technology_similarity
expert_similarity_label,,,,,,,,
0,0.039694,0.193013,0.342632,0.307135,0.139939,0.020921,0.098087,0.080593
1,0.327994,0.482277,0.646867,0.629925,0.136520,0.523880,0.521740,0.071780
2,0.630855,0.772853,0.959423,0.959216,0.122338,1.000000,0.928016,0.091555
3,0.629499,0.770281,0.958281,0.958068,0.134790,1.000000,0.925990,0.095383
4,0.629894,0.769071,0.952027,0.951782,0.111586,1.000000,0.914894,0.075091
